# 🇦🇺🇺🇸 US & ASX AI Quantitative Research Platform — Google Colab 主工作台

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidshizw/stock-colab/blob/main/notebooks/01_Colab_Quickstart_End_to_End_Pipeline.ipynb)

本 Notebook 专为 **Google Colab** 环境设计，覆盖从 **美股（NYSE/Nasdaq）与澳股（ASX 200）数据入库、DuckDB + Parquet 持久化（挂载 Google Drive）、6 大类量化因子计算、跨市场隔夜联动（US Overnight → ASX）、含澳洲 Franking Credit 与美股预提税的组合优化与回测、AI 多智能体报告** 到 **Colab 内嵌交互式看板** 的完整闭环。

In [ ]:
# 1. 环境初始化：支持私有/公开 GitHub 仓库自动拉取 + Google Drive 缓存持久化
import os, subprocess, sys
from pathlib import Path

REPO_SLUG = 'davidshizw/stock-colab'
if 'google.colab' in sys.modules:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    drive_repo = Path('/content/drive/MyDrive/stock-colab')
    local_repo = Path('/content/stock-colab')

    if not Path('pyproject.toml').exists():
        # 优先使用 Colab Secrets 中的 GITHUB_TOKEN 拉取私有仓库并同步至 Google Drive
        gh_token = None
        try:
            gh_token = userdata.get('GITHUB_TOKEN')
        except Exception:
            pass
        if not gh_token and not drive_repo.exists() and not local_repo.exists():
            from getpass import getpass
            gh_token = getpass('私有仓库首次拉取请输入 GitHub Personal Access Token (之后将自动缓存在 Google Drive): ').strip()

        clone_url = f'https://{gh_token}@github.com/{REPO_SLUG}.git' if gh_token else f'https://github.com/{REPO_SLUG}.git'
        target_dir = drive_repo if Path('/content/drive/MyDrive').exists() else local_repo
        if not (target_dir / 'pyproject.toml').exists():
            print(f'Cloning {REPO_SLUG} into {target_dir} ...')
            subprocess.run(['git', 'clone', clone_url, str(target_dir)], check=False)
        else:
            subprocess.run(['git', '-C', str(target_dir), 'pull'], check=False)
        if (target_dir / 'pyproject.toml').exists():
            os.chdir(target_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=False)

if Path('src').exists() and str(Path('src').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('src').resolve()))

from quant_platform.colab.runtime import ColabSessionManager
colab_mgr = ColabSessionManager(drive_folder_name='stock_colab_store', monthly_cu_budget=1000.0)
print('Colab Compute & Storage Status:', colab_mgr.get_compute_unit_status())

In [ ]:
# 2. 初始化 DuckDB + Parquet 存储并执行美股与澳股数据入库
# 支持 mode='fixture' (高保真确定性跨市场测试数据) 或 mode='yfinance' (实时拉取 Yahoo Finance 数据)
from datetime import date, timedelta
from quant_platform.data.store import MarketDataStore
from quant_platform.data.ingestion import build_default_pipeline

store = MarketDataStore(db_path=colab_mgr.duckdb_path, parquet_dir=colab_mgr.parquet_dir)
DATA_MODE = 'fixture'  # 可改为 'yfinance' 在 Colab 中拉取最新实时行情

end_date = date(2026, 11, 6)
start_date = end_date - timedelta(days=500)

pipeline = build_default_pipeline(store, mode=DATA_MODE)
ing_res = pipeline.run_ingestion(start_date=start_date, end_date=end_date, incremental=False)
print('Ingestion Summary:', ing_res)

In [ ]:
# 3. 量化因子计算与美股 (US) / 澳股 (ASX) 系统化选股排名
import pandas as pd
from quant_platform.factors.engine import FactorEngine
from quant_platform.screener.ranker import EquityScreener

factor_engine = FactorEngine(store)
screener = EquityScreener(store, factor_engine)

us_top = screener.screen(as_of_date=end_date, market='US', top_n=8)
asx_top = screener.screen(as_of_date=end_date, market='ASX', top_n=8)

print('=== 美股 (US) 多因子排名 Top 8 ===')
display(pd.DataFrame(us_top)[['rank','symbol','name','sector','close','pe_ratio','roe','mom_6m','composite_score']])

print('\n=== 澳股 (ASX) 多因子排名 Top 8 (含澳洲 Franking Credit 税前等效股息率) ===')
display(pd.DataFrame(asx_top)[['rank','symbol','name','sector','close','pe_ratio','dividend_yield','franking_rate','franking_grossed_yield','composite_score']])

In [ ]:
# 4. 跨市场联动分析 (US Overnight -> ASX Session) 与因子 IC / 分层检验
from quant_platform.cross_market.us_asx_analyzer import CrossMarketAnalyzer
from quant_platform.factors.evaluation import FactorEvaluator

cm = CrossMarketAnalyzer(store).analyze_overnight_transmission(start_date=start_date, end_date=end_date)
print(f"Zero Look-Ahead Alignment Verified: {cm['alignment_verified']} (Violations: {cm['lookahead_violations']})")
display(pd.DataFrame(cm['sector_sensitivity']))

ev = FactorEvaluator(store, factor_engine).evaluate_factors(start_date=start_date, end_date=end_date, market='ALL')
print('\n=== 因子 IC / Rank IC / 衰减与换手评估 ===')
display(pd.DataFrame(ev['factor_summary']))

In [ ]:
# 5. 策略回测、组合优化与 Colab 内嵌 ECharts 交互式看板渲染
from IPython.display import HTML, display
from quant_platform.backtest.engine import BacktestEngine
from quant_platform.schemas import BacktestConfig, CurrencyCode
from quant_platform.agents.coordinator import ResearchCoordinatorAgent

bt_cfg = BacktestConfig(
    strategy_name='US+ASX Multi-Factor Composite',
    market_filter='ALL',
    base_currency=CurrencyCode.AUD,
    start_date=start_date + timedelta(days=60),
    end_date=end_date,
    rebalance_frequency='M',
    top_n=6,
    weighting_scheme='risk_parity',
    include_franking_credits=True,
    apply_us_withholding_tax=True,
)
bt_res = BacktestEngine(store, factor_engine).run_backtest(bt_cfg)

# 在 Colab 单元格内直接渲染交互式 ECharts 投研看板
html_widget = colab_mgr.render_inline_colab_dashboard(
    us_top=us_top,
    asx_top=asx_top,
    backtest_result=bt_res.model_dump(mode='json'),
    cross_market=cm,
)
display(HTML(html_widget))

# 生成 AI 多智能体确定性溯源报告
report = ResearchCoordinatorAgent(store).generate_research_report(as_of_date=end_date, market_scope='ALL', focus_symbol='BHP.AX')
print('\n=== AI Research Coordinator Executive Summary ===')
print(report.executive_summary)

In [ ]:
# 6. (可选) 在 Google Colab 内部以后台线程启动完整 10 模块 FastAPI 投研看板并通过 Colab IFrame 访问
# 在 Colab 中取消下方注释即可打开完整全功能看板：
# import threading, uvicorn
# from quant_platform.api.app import create_app
# app = create_app(store)
# threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'), daemon=True).start()
# if 'google.colab' in sys.modules:
#     from google.colab import output
#     output.serve_kernel_port_as_iframe(8000, height='850')